# Pipeline 3 (HOG+SVM + Pinhole) & Final Ortak Değerlendirme
**Sorumlu:** Kişi 3 (Ali) | **Branch:** `feat/pipeline3-hog-pinhole` (ve merge sonrası `main`)

Bu notebook iki aşamalıdır:
1. **Aşama A (Kendi Branch'inde):** Klasik Baseline (`HOG + Linear SVM` Pedestrian dedektörü + `Pinhole Camera Model`) eğitilir ve ortak Drive'a kaydedilir.
2. **Aşama B (Tüm Branch'ler `main`'de Merge Edildikten Sonra):** Ortak Drive'daki tüm ağırlıklar (`yolov10n`, `yolov8n`, `hog_svm`) çekilerek **tek bir GPU üzerinde** tüm test seti değerlendirmesi (`04_evaluate_all.py --multi-seed-summary`) ve hız benchmark'ı (`05_benchmark_speed.py`) çalıştırılır.

In [ ]:
!nvidia-smi
from google.colab import drive
import os

drive.mount('/content/drive')
SHARED_DRIVE = '/content/drive/MyDrive/CS423_Shared_Project'
os.makedirs(f'{SHARED_DRIVE}/checkpoints', exist_ok=True)
os.makedirs(f'{SHARED_DRIVE}/results', exist_ok=True)
print('Ortak Drive klasörü hazır:', SHARED_DRIVE)

In [ ]:
REPO_URL = "https://github.com/nehirvardar/cs423-assistive-navigation.git"
BRANCH = "pipeline-3"  # Final birleştirme aşamasında "main" yapın

!rm -rf /content/project
!git clone -b {BRANCH} {REPO_URL} /content/project
%cd /content/project
!pip install -q -r requirements.txt
!apt-get update -qq && apt-get install -qq -y aria2

In [ ]:
# Tam KITTI indirmesi ve hazırlığı
!python scripts/download_kitti.py
!python scripts/01_prepare_data.py

In [ ]:
# AŞAMA A: Klasik HOG + Linear SVM Pedestrian dedektörünü eğit ve Drive'a kaydet
!python scripts/02_train_hog.py
!cp -v checkpoints/hog_svm_pedestrian.joblib {SHARED_DRIVE}/checkpoints/

In [ ]:
# AŞAMA B (Merge Sonrası Final): Ortak Drive'daki tüm ekip ağırlıklarını çek
!mkdir -p checkpoints
!cp -v {SHARED_DRIVE}/checkpoints/* checkpoints/

# Tüm test seti üzerinde 3 Pipeline karşılaştırması + 3-seed mean+/-std + BBox Ablasyonu
!python scripts/04_evaluate_all.py --split test --multi-seed-summary --device cuda

# Tek GPU üzerinde adil Latency (ms) ve FPS ölçümü
!python scripts/05_benchmark_speed.py --warmup 20 --runs 100 --device cuda

# Final rapor tablolarını ve grafiklerini ortak Drive'a kaydet
!cp -r results {SHARED_DRIVE}/